# market-forecast Colab Demo
This notebook runs a minimal end-to-end test of the `market-forecast` package.

In [ ]:
# Install mode options:
#   GITHUB  -> install directly from a GitHub URL
#   EDITABLE_LOCAL -> install from a local clone in Colab (for example /content/market_forecast_model)
INSTALL_MODE = "GITHUB"  # change to "EDITABLE_LOCAL" when working from a local clone
GITHUB_REPO = "https://github.com/your-org/market_forecast_model.git"
LOCAL_REPO_PATH = "/content/market_forecast_model"

import os
import subprocess
import sys

def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", *args])

if INSTALL_MODE == "EDITABLE_LOCAL":
    if not os.path.isdir(LOCAL_REPO_PATH):
        raise FileNotFoundError(
            f"Local repo path not found: {LOCAL_REPO_PATH}. Clone the repo first or switch INSTALL_MODE to GITHUB."
        )
    pip_install("-e", LOCAL_REPO_PATH)
else:
    pip_install(f"git+{GITHUB_REPO}")


In [ ]:
import numpy as np
import pandas as pd

from market_forecast.config.schemas import ForecastConfig
from market_forecast.pipelines.forecast_pipeline import ForecastPipeline

In [ ]:
idx = pd.date_range('2021-01-01', periods=350, freq='B')
rng = np.random.default_rng(42)
spy = 100 * np.cumprod(1 + rng.normal(0.0005, 0.01, size=len(idx)))
qqq = 80 * np.cumprod(1 + rng.normal(0.0004, 0.011, size=len(idx)))
prices = pd.DataFrame({'SPY': spy, 'QQQ': qqq}, index=idx)
prices.tail()

In [ ]:
config = ForecastConfig()
pipeline = ForecastPipeline(config).fit(prices, target_col='SPY')
pred = pipeline.predict([1,2,4,8,12])
signals = pipeline.generate_signals(pred)
signals

In [ ]:
state = pipeline.summarize_current_state()
state